# Advanced Robotics Project, Week 1
## Notebook 1: Where is the hand? Forward and inverse kinematics

Obuda University, Antal Bejczy Center for Intelligent Robotics

A robot arm knows its joint angles, but a task is always given as a place for the tool. This notebook builds the two translations between the two:

- **forward kinematics (FK):** from the joint angles to the pose of the tool,
- **inverse kinematics (IK):** from a wanted tool position back to the joint angles.

Parts A and B work in **2D**, in the plane, where everything can be drawn and checked by hand. Part C moves to **3D** with a real six-joint arm, the UR5. Parts A and B have two one-line exercises each; Part C is a demo.

**How to work in this notebook.** Run the cells from top to bottom with Shift+Enter. Four cells contain a line for you, between these two markers:

    # ---- your code: 1 line ----
    # ----------------------------

Change only that line, then run the cell starting with `# Check` right below it. It prints *Correct*, or *Not yet* with a hint. Part C has nothing to fill in. If something gets stuck, use Runtime → Restart session, then Run all. Everything used here is already installed in Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider
from IPython.display import HTML

np.set_printoptions(precision=3, suppress=True)

---
# Part A: poses in the plane (2D)

A **pose** says where something is and which way it faces. In the plane that is a position $(x, y)$ and one angle $\theta$.

A rotation by $\theta$ is stored as a $2 \times 2$ matrix. Its columns are the turned $x$ and $y$ axes, written in the old frame:

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$$

Two of its properties have names worth knowing.

- **Transpose.** `R.T` is the transposed matrix $R^\top$: its rows become columns. For a rotation, the transpose is also the **inverse**, $R^\top R = I$, so undoing a rotation costs nothing. The same equation says that lengths and angles are kept: the body stays rigid.
- **Determinant.** A rotation has determinant $\det R = +1$, which means left and right are kept. A matrix with determinant $-1$ is a **reflection**: it turns a shape into its mirror image, something no rigid motion can do.

A **homogeneous transform** packs the rotation $R$ and the position $p$ into one $3 \times 3$ matrix, so that changing frames is a single multiplication:

$$T = \begin{pmatrix} R & p \\ 0 \;\; 0 & 1 \end{pmatrix}$$

The set of all rotations in the plane is called **SO(2)**, the special orthogonal group; the set of all poses in the plane, rotation plus position, is **SE(2)**, the special Euclidean group. Part C uses their 3D versions, SO(3) and SE(3).

In NumPy, `@` is matrix multiplication. The plain `*` would multiply element by element, which is something else.

In [ ]:
def rot2(theta):
    '''2x2 rotation matrix, an element of SO(2).'''
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s],
                     [s,  c]])

def se2(theta, x=0.0, y=0.0):
    '''3x3 homogeneous transform: turned by theta [rad], positioned at (x, y) [m]. An element of SE(2).'''
    T = np.eye(3)
    T[:2, :2] = rot2(theta)
    T[:2, 2] = [x, y]
    return T

R = rot2(np.deg2rad(30))
print("R =\n", R)
print("R.T @ R =\n", R.T @ R)
print("R.T is the inverse of R:", np.allclose(R.T, np.linalg.inv(R)))
print("det(R) =", round(np.linalg.det(R), 6))

What the sign of the determinant means, on an F-shaped plate: the same shape rotated by 60° (determinant $+1$) and multiplied by `[[1, 0], [0, -1]]` (determinant $-1$). The rotated F still reads as an F. The other one is its mirror image, and no rotation turns it back.

In [ ]:
F = np.array([[0, 0], [0, 2], [1.2, 2], [1.2, 1.6], [0.4, 1.6], [0.4, 1.2],
              [0.9, 1.2], [0.9, 0.8], [0.4, 0.8], [0.4, 0], [0, 0]])
turn60 = rot2(np.deg2rad(60))
mirror = np.array([[1, 0],
                   [0, -1]])
print("det(turn60) =", round(np.linalg.det(turn60), 3), "   det(mirror) =", round(np.linalg.det(mirror), 3))

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, (title, A) in zip(axes, [("original", np.eye(2)), ("rotated by 60°, det = +1", turn60), ("mirrored, det = -1", mirror)]):
    P = F @ A.T                     # apply A to every corner point
    ax.fill(P[:, 0], P[:, 1], color="#378ADD", alpha=0.6)
    ax.plot(P[:, 0], P[:, 1], color="#333", lw=1)
    ax.set_title(title)
    ax.set_aspect("equal"); ax.grid(alpha=.3)
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
plt.show()

### Order matters

`se2(theta)` alone is a pure turn, and `se2(0.0, L, 0.0)` is a pure step of `L` metres along $x$. Multiplying the two combines the moves, but the order changes the result: **matrix multiplication is not commutative**, and `A @ B` is in general a different matrix from `B @ A`. Read `turn @ step` from left to right: first turn, then step along the turned $x$ axis.

In [ ]:
turn = se2(np.deg2rad(90))      # turn by 90 degrees
step = se2(0.0, 1.0, 0.0)       # 1 m along x
print("turn @ step ends at", (turn @ step)[:2, 2].round(2))
print("step @ turn ends at", (step @ turn)[:2, 2].round(2))

### The planar arm

The arm has three revolute joints with angles $\theta_1, \theta_2, \theta_3$ (the list `q`) and three links of lengths $L_1, L_2, L_3$ (the list `L`). To find the tool, walk from the base outwards: at every joint, turn by the joint angle, then go along the link.

### Exercise A1: one joint

One joint is exactly the `turn @ step` from the cell above, with the joint angle and the link length in place of 90° and 1 m. Written the other way round, or as `se2(theta, L, 0.0)`, the step would go along the old $x$ axis instead.

Example: with `theta` = 90° and `L` = 1 m the joint ends at (0, 1), facing up.

Your line: `T_i` is `se2(theta)` multiplied by `se2(0.0, L, 0.0)`, in this order.

In [ ]:
def joint_transform(theta, L):
    '''Transform of one joint: turn by theta [rad], then go L [m] along the link.'''
    # ---- your code: 1 line ----
    T_i = None
    # ----------------------------
    if T_i is None:
        return np.eye(3)   # not written yet: no motion, so the later cells still run
    return T_i

In [ ]:
# Check
try:
    got = np.asarray(joint_transform(np.deg2rad(90), 1.0), dtype=float)
    if got.shape != (3, 3):
        print("Not yet: joint_transform should return a 3x3 matrix. Put @ between the two se2(...) calls.")
    elif np.allclose(got, np.eye(3)):
        print("Not yet: T_i is still None. Write your line between the two markers.")
    elif np.allclose(got[:2, 2], [0.0, 1.0]) and np.allclose(got[:2, :2], rot2(np.deg2rad(90))):
        print("Correct: a 90° turn followed by 1 m along the turned x axis ends at (0, 1).")
    elif np.allclose(got, se2(np.deg2rad(90)) * se2(0.0, 1.0, 0.0)):
        print("Not yet: * multiplies element by element. Matrix multiplication is @.")
    elif np.allclose(got[:2, 2], [1.0, 0.0]):
        print("Not yet: the joint ends at (1, 0), so the step went along the old x axis. The turn comes first: se2(theta) @ se2(0.0, L, 0.0).")
    else:
        print("Not yet: for 90° and 1 m the joint should end at (0, 1); yours ends at", np.round(got[:2, 2], 2))
except Exception as e:
    print("Not yet: the code raised an error:", e)

Here is what the wrong order does to a whole arm. Joint 1 is drawn at 0°, 30°, 60° and 90°. On the right, every joint steps before it turns, and the first link never moves.

In [ ]:
def arm_points(q, L, turn_first):
    T = np.eye(3)
    points = [T[:2, 2].copy()]
    for theta, Li in zip(q, L):
        T = T @ (se2(theta) @ se2(0.0, Li, 0.0) if turn_first else se2(theta, Li, 0.0))
        points.append(T[:2, 2].copy())
    return np.array(points)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
sweep = [0, 30, 60, 90]
for ax, turn_first, title in zip(axes, (True, False), ("turn, then go along the link", "go along the link, then turn")):
    for k, t1 in enumerate(sweep):
        p = arm_points(np.deg2rad([t1, 45, -20]), (1.0, 0.8, 0.5), turn_first)
        ax.plot(p[:, 0], p[:, 1], "-o", lw=3, ms=5, color="#333", alpha=1.0 if k == len(sweep) - 1 else 0.25)
    ax.plot(0, 0, "s", ms=10, color="#555")
    ax.set_title(title)
    ax.set_aspect("equal"); ax.grid(alpha=.3)
    ax.set_xlim(-1.2, 2.5); ax.set_ylim(-0.4, 2.3)
plt.show()

### Exercise A2: chain the joints

`fk_3r` walks from the base to the tool. `T` starts as the identity matrix (we stand at the base and nothing has moved), and at every joint the pose so far is multiplied by that joint's transform, with the new joint on the right. After the third joint, `T` is the pose of the tool as seen from the base.

On the way the loop also fills the list `points` with the origin of every frame: the base, joint 2, joint 3 and the tool. `draw_arm` later connects these four points with lines.

Example: with the angles (30°, 45°, −20°) and the links (1.0, 0.8, 0.5) m the tool ends at (1.36, 1.68) m.

Your line: the new `T` is the old `T` multiplied by `T_i`, in this order.

In [ ]:
def fk_3r(q, L=(1.0, 0.8, 0.5)):
    '''Forward kinematics of the planar arm. Returns points (4x2: base, joint 2, joint 3, tool) and T (3x3 tool pose).'''
    T = np.eye(3)
    points = [T[:2, 2].copy()]
    for theta, Li in zip(q, L):
        T_i = joint_transform(theta, Li)
        # ---- your code: 1 line ----
        pass
        # ----------------------------
        points.append(T[:2, 2].copy())
    return np.array(points), T

In [ ]:
# Check
try:
    q_test, L_test = np.deg2rad([30, 45, -20]), (1.0, 0.8, 0.5)
    angles = np.cumsum(q_test)
    want = np.array([np.dot(L_test, np.cos(angles)), np.dot(L_test, np.sin(angles))])
    a1_ok = np.allclose(np.asarray(joint_transform(np.deg2rad(90), 1.0), dtype=float)[:2, 2], [0.0, 1.0])
    _, T_test = fk_3r(q_test, L_test)
    T_test = np.asarray(T_test, dtype=float)
    got = T_test[:2, 2]
    T_rev, T_elem = np.eye(3), np.eye(3)
    for th, l in zip(q_test, L_test):
        T_rev = joint_transform(th, l) @ T_rev
        T_elem = T_elem * joint_transform(th, l)
    if not a1_ok:
        print("Not yet: finish Exercise A1 first, then run this check again.")
    elif np.allclose(got, want):
        print(f"Correct: the tool is at ({got[0]:.2f}, {got[1]:.2f}) m.")
    elif np.allclose(T_test, np.eye(3)):
        print("Not yet: T never changes, so the tool stays at the base. Update T inside the loop.")
    elif np.allclose(T_test, T_rev):
        print("Not yet: this is T_i @ T. The new joint goes on the right: T @ T_i.")
    elif np.allclose(T_test, T_elem):
        print("Not yet: * multiplies element by element. Matrix multiplication is @.")
    else:
        print("Not yet: the tool should be at", np.round(want, 2), "but yours is at", np.round(got, 2))
except Exception as e:
    print("Not yet: the code raised an error:", e)

In [ ]:
def draw_arm(points, T_tool=None, ax=None, target=None, title=""):
    '''Draw the planar arm; optionally the tool frame (red: x axis, blue: y axis) and a target.'''
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.plot(points[:, 0], points[:, 1], "-o", lw=4, ms=8, color="#333")
    ax.plot(points[0, 0], points[0, 1], "s", ms=12, color="#555")
    if T_tool is not None:
        o, R = T_tool[:2, 2], T_tool[:2, :2]
        ax.arrow(*o, *(0.25 * R[:, 0]), color="#D85A30", width=0.012)
        ax.arrow(*o, *(0.25 * R[:, 1]), color="#378ADD", width=0.012)
    if target is not None:
        ax.plot(*target, "*", ms=18, color="#2E9E5B")
    ax.set_aspect("equal"); ax.grid(alpha=.3)
    ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6)
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_title(title)
    return ax

points, T_tool = fk_3r(np.deg2rad([30, 45, -20]))
print("tool position:", T_tool[:2, 2])
draw_arm(points, T_tool, title="q = (30°, 45°, -20°)")
plt.show()

Move the sliders; the drawing uses your `fk_3r`. The red arrow is the tool's $x$ axis, the blue one its $y$ axis. In the plane the tool's angle is simply $\theta_1 + \theta_2 + \theta_3$. In 3D, rotations do not add up like that.

In [ ]:
def show_fk(t1=30, t2=45, t3=-20):
    points, T = fk_3r(np.deg2rad([t1, t2, t3]))
    draw_arm(points, T, title=f"tool at ({T[0, 2]:.2f}, {T[1, 2]:.2f}) m, facing {np.rad2deg(np.arctan2(T[1, 0], T[0, 0])):.0f}°")
    plt.show()

interact(show_fk,
         t1=FloatSlider(value=30, min=-180, max=180, step=1, description="theta1"),
         t2=FloatSlider(value=45, min=-180, max=180, step=1, description="theta2"),
         t3=FloatSlider(value=-20, min=-180, max=180, step=1, description="theta3"));

---
# Part B: inverse kinematics in the plane (2D)

Forward kinematics always has exactly one answer. Inverse kinematics runs the other way (the tool should be at $(x, y)$: which joint angles put it there?) and can have two answers, one, or none. We use a two-link arm with $L_1$ = 1.0 m and $L_2$ = 0.8 m.

### Exercise B1: the elbow angle

Base, elbow and tool form a triangle, and we know all three sides: $L_1$, $L_2$, and the distance from the base to the target. The law of cosines then gives the cosine of the elbow angle directly:

$$\cos\theta_2 = \frac{x^2 + y^2 - L_1^2 - L_2^2}{2 \, L_1 \, L_2}$$

A direct formula like this is what a **closed-form solution** means. In Python a square is written `x**2`.

Example: for the target (1.2, 0.6), $\cos\theta_2 = (1.44 + 0.36 - 1.00 - 0.64) / 1.6 = 0.10$.

Your line: `c2` is the formula above, with the top and the bottom each in parentheses.

In [ ]:
def cos_theta2(x, y, L1=1.0, L2=0.8):
    '''cos(theta2) for a two-link arm whose tool should be at (x, y).'''
    # ---- your code: 1 line ----
    c2 = None
    # ----------------------------
    return c2

In [ ]:
# Check
try:
    got = cos_theta2(1.2, 0.6)
    if got is None:
        print("Not yet: c2 is still None. Write your line between the two markers.")
    elif np.isclose(float(got), 0.1) and np.isclose(float(cos_theta2(1.8, 0.0)), 1.0):
        print("Correct: cos(theta2) is 0.10 for (1.2, 0.6), and exactly 1 for the fully stretched arm at (1.8, 0).")
    elif np.isclose(float(got), 0.4):
        print("Not yet: only L2**2 got divided. Put the whole top part in parentheses.")
    elif np.isclose(float(got), 0.064):
        print("Not yet: the bottom needs parentheses too: / (2 * L1 * L2).")
    else:
        print(f"Not yet: for (1.2, 0.6) the value should be 0.10, yours is {float(got):.3f}. Check the signs and the squares.")
except Exception as e:
    print("Not yet: the code raised an error:", e)

### Exercise B2: can the arm reach the target?

A cosine always lies between $-1$ and $1$. If the formula gives something outside that range, no elbow angle has that cosine: the target is outside the **workspace**, and IK has no answer. Inside the range there are two answers, $\theta_2$ and $-\theta_2$, the elbow-down and the elbow-up arm. The code for both is already written below your line.

Two pieces of Python do the test. `abs(c2)` is the **absolute value** of `c2`, the number without its sign. And a comparison is itself a value: `abs(c2) <= 1.0` is either `True` or `False`. So `reachable = abs(c2) <= 1.0` takes $|c_2|$, compares it with 1, and stores the answer in `reachable`. `True` means that a real angle with this cosine exists.

Example: `c2` = 0.10 gives `True`; `c2` = 1.9 gives `False`.

Your line: `reachable` is the comparison "the absolute value of `c2` is at most 1".

In [ ]:
def ik_2r(target, L1=1.0, L2=0.8):
    '''IK of the two-link arm: [(theta1, theta2) elbow-down, (theta1, theta2) elbow-up], or None if out of reach.'''
    x, y = target
    c2 = cos_theta2(x, y, L1, L2)
    if c2 is None:
        return []          # Exercise B1 is not done yet
    # ---- your code: 1 line ----
    reachable = None
    # ----------------------------
    if reachable is None:
        return []          # Exercise B2 is not done yet
    if not reachable:
        return None        # outside the workspace: no answer
    answers = []
    for sign in (+1, -1):  # +1: elbow-down, -1: elbow-up
        t2 = sign * np.arccos(np.clip(c2, -1.0, 1.0))
        t1 = np.arctan2(y, x) - np.arctan2(L2 * np.sin(t2), L1 + L2 * np.cos(t2))
        answers.append((t1, t2))
    return answers

In [ ]:
# Check
try:
    def tool_2r(t1, t2, L1=1.0, L2=0.8):
        return np.array([L1*np.cos(t1) + L2*np.cos(t1 + t2), L1*np.sin(t1) + L2*np.sin(t1 + t2)])
    if cos_theta2(1.2, 0.6) is None:
        print("Not yet: finish Exercise B1 first.")
    else:
        inside, too_far, too_close = ik_2r((1.2, 0.6)), ik_2r((2.5, 1.0)), ik_2r((0.1, 0.0))
        if inside == []:
            print("Not yet: reachable is still None. Write your line between the two markers.")
        elif inside is None:
            print("Not yet: (1.2, 0.6) is reachable, but ik_2r said no. The comparison is the wrong way round: at most 1 means <=.")
        elif too_far is not None:
            print("Not yet: (2.5, 1.0) is farther than 1.8 m, yet ik_2r returned arms. reachable must be False when abs(c2) is above 1.")
        elif too_close is not None:
            print("Not yet: (0.1, 0.0) is too close to the base, where c2 is below -1. Compare the absolute value, abs(c2), not c2 itself.")
        elif len(inside) == 2 and all(np.allclose(tool_2r(*s), (1.2, 0.6)) for s in inside):
            print("Correct: two answers inside the workspace, none outside it.")
        else:
            print("Not yet: the test looks right, but the arms miss the target. Check Exercise B1.")
except Exception as e:
    print("Not yet: the code raised an error:", e)

Three targets first, then one you can move. The grey circles mark the edges of the workspace: the arm reaches everything between them.

In [ ]:
for target in [(1.2, 0.6), (1.7, 0.0), (2.5, 1.0)]:
    answers = ik_2r(target)
    if answers is None:
        print(target, "-> no answer: outside the workspace")
    elif len(answers) == 0:
        print(target, "-> finish Exercises B1 and B2 first")
    else:
        print(target, "-> elbow-down [deg]:", np.rad2deg(answers[0]).round(1), "   elbow-up [deg]:", np.rad2deg(answers[1]).round(1))

In [ ]:
def show_ik(x=1.2, y=0.6, L1=1.0, L2=0.8):
    answers = ik_2r((x, y), L1, L2)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    th = np.linspace(0, 2*np.pi, 200)
    for r, style in [(L1 + L2, "-"), (abs(L1 - L2), "--")]:
        ax.plot(r*np.cos(th), r*np.sin(th), style, color="#aaa", lw=1)
    if answers is None:
        ax.set_title("no answer: the target is outside the workspace")
    elif len(answers) == 0:
        ax.set_title("finish Exercises B1 and B2 to see the arm")
    else:
        for (t1, t2), col in zip(answers, ["#333", "#D85A30"]):
            elbow = (L1*np.cos(t1), L1*np.sin(t1))
            tool = (elbow[0] + L2*np.cos(t1 + t2), elbow[1] + L2*np.sin(t1 + t2))
            ax.plot([0, elbow[0], tool[0]], [0, elbow[1], tool[1]], "-o", lw=4, ms=7, color=col, alpha=.85)
        ax.set_title("elbow-down (black) and elbow-up (orange)")
    ax.plot(x, y, "*", ms=18, color="#2E9E5B")
    ax.set_aspect("equal"); ax.grid(alpha=.3)
    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2)
    plt.show()

interact(show_ik,
         x=FloatSlider(value=1.2, min=-2, max=2, step=0.05),
         y=FloatSlider(value=0.6, min=-2, max=2, step=0.05));

---
# Part C: a real arm in 3D (demo)

From here on we are in **3D**, and there is nothing to fill in: run the cells and look. The arm is a UR5, a six-joint arm made by Universal Robots.

A pose in 3D is a $4 \times 4$ matrix: a $3 \times 3$ rotation from SO(3) and a position, together an element of SE(3). Forward kinematics is the same walk as in Part A, one transform per joint, multiplied in order.

### Denavit–Hartenberg parameters

Between two frames in 3D, a general transform needs six numbers: three for the position and three for the orientation. The **Denavit–Hartenberg (DH) convention** places the frame of every joint in a particular way, so that four numbers per joint are enough:

| | meaning |
|---|---|
| $\theta$ (theta) | the joint angle: a turn about the joint axis $z$. This is the number that changes when the joint moves. |
| $d$ | an offset along the same $z$ axis |
| $a$ | the link length: a slide along the new $x$ axis |
| $\alpha$ (alpha) | the twist: a tilt about that $x$ axis, which lines $z$ up with the next joint axis |

Read every row of the table as "turn and slide along $z$, then slide and tilt along $x$". `dh_transform` turns one row into a $4 \times 4$ matrix, and `fk_ur5` multiplies the six of them, exactly as `fk_3r` did in 2D.

A warning for later: datasheets use two versions, **standard** and **modified** DH, which place the frames differently, so their numbers are not interchangeable. The UR5 values below are standard DH.

In [ ]:
UR5_D     = np.array([0.089159, 0.0,     0.0,      0.10915, 0.09465, 0.0823])
UR5_A     = np.array([0.0,     -0.425,  -0.39225,  0.0,     0.0,     0.0])
UR5_ALPHA = np.array([np.pi/2,  0.0,     0.0,      np.pi/2, -np.pi/2, 0.0])

print("UR5, standard DH parameters")
print("joint   theta      d [m]      a [m]   alpha [deg]")
for i in range(6):
    print(f"  {i + 1}      q{i + 1}   {UR5_D[i]:9.5f}  {UR5_A[i]:9.5f}   {np.rad2deg(UR5_ALPHA[i]):6.0f}")

def dh_transform(theta, d, a, alpha):
    '''4x4 transform of one DH row: turn theta and slide d along z, then slide a and tilt alpha along x.'''
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([[ct, -st*ca,  st*sa, a*ct],
                     [st,  ct*ca, -ct*sa, a*st],
                     [0.,     sa,     ca,    d],
                     [0.,     0.,     0.,   1.]])

def fk_ur5(q):
    '''UR5 forward kinematics. Returns points (7x3 frame origins) and T (4x4 tool pose).'''
    T = np.eye(4)
    points = [T[:3, 3].copy()]
    for i in range(6):
        T = T @ dh_transform(q[i], UR5_D[i], UR5_A[i], UR5_ALPHA[i])
        points.append(T[:3, 3].copy())
    return np.array(points), T

q_home = np.array([0.0, -np.pi/2, np.pi/2, -np.pi/2, -np.pi/2, 0.0])
_, T_home = fk_ur5(q_home)
print("\ntool position in the home pose:", T_home[:3, 3].round(3), "m")

In [ ]:
def plot_ur5(q, target=None, title="UR5"):
    '''Interactive 3D drawing of the UR5 (drag to rotate).'''
    points, T = fk_ur5(q)
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode="lines+markers",
                               line=dict(width=10, color="#333"), marker=dict(size=5, color="#D85A30"), name="links"))
    o, R = T[:3, 3], T[:3, :3]
    for k, col in zip(range(3), ["#D85A30", "#2E9E5B", "#378ADD"]):   # tool x, y, z axes
        e = o + 0.12 * R[:, k]
        fig.add_trace(go.Scatter3d(x=[o[0], e[0]], y=[o[1], e[1]], z=[o[2], e[2]], mode="lines",
                                   line=dict(width=6, color=col), showlegend=False))
    if target is not None:
        fig.add_trace(go.Scatter3d(x=[target[0]], y=[target[1]], z=[target[2]], mode="markers",
                                   marker=dict(size=8, color="#2E9E5B", symbol="diamond"), name="target"))
    fig.update_layout(title=title, height=520, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data", xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]"))
    return fig

plot_ur5(q_home, title="UR5 in its home pose (drag to rotate)").show()

### The Jacobian

Move one joint by a tiny angle and the tool moves a tiny bit in some direction. The **Jacobian** $J$ collects these directions in a table. For the position of the UR5 tool it has 3 rows and 6 columns, and the next cell builds it exactly like that: it nudges each joint in turn and records where the tool went.

- A **column** belongs to one joint. It says in which direction, and how fast, the tool moves when only that joint turns (metres per radian).
- A **row** belongs to one direction of the tool, $x$, $y$ or $z$. It says how much each of the six joints contributes to motion that way.

In [ ]:
def jacobian_pos(q, eps=1e-6):
    '''3x6 position Jacobian by nudging: column j is how the tool moves when joint j turns.'''
    J = np.zeros((3, 6))
    for j in range(6):
        dq = np.zeros(6)
        dq[j] = eps
        _, T_plus = fk_ur5(q + dq)
        _, T_minus = fk_ur5(q - dq)
        J[:, j] = (T_plus[:3, 3] - T_minus[:3, 3]) / (2 * eps)
    return J

J = jacobian_pos(q_home)
print("J in the home pose [m/rad]\n")
print("       " + "".join(f"joint {j + 1}   " for j in range(6)))
for name, row in zip(["x", "y", "z"], J):
    print(f"  {name}   " + "".join(f"{(0.0 if abs(v) < 5e-4 else v):7.3f}   " for v in row))
print("\nrank of J:", np.linalg.matrix_rank(J))

**Reading the numbers.** In the home pose the tool is at $x = -0.487$, $y = -0.109$, $z = 0.432$ m.

- **Column 1**, the base joint, is $(0.109, -0.487, 0)$. The base turns about the vertical axis, so the tool swings sideways along a horizontal circle, and the $z$ entry is zero.
- **Columns 2 and 3**, shoulder and elbow, both have $z = -0.487$: turning either joint moves the tool down. Column 2 also moves it along $x$ ($-0.343$).
- **Column 5** is small, $(0, -0.082, 0)$: the tool sits only 0.082 m from this wrist axis (the last DH offset $d$), so turning it barely moves the tool point.
- **Column 6** is all zeros. The last joint spins the tool about its own axis, and a point on that axis does not move. It still changes the tool's orientation, which this position-only table does not show.
- **Rank 3.** Together the six columns still point in all three directions, so from this pose the tool can move any way. At a **singularity** the rank drops below 3, and one direction is missing from the table.

### Inverse kinematics by nudging

For the two-link arm a formula gave the answer. The UR5 also has a closed-form solution, but it runs to pages, and a general arm has none. So we solve IK with a loop:

1. measure the gap between the tool and the target,
2. turn the gap into a joint move with the **pseudo-inverse** of the Jacobian, `np.linalg.pinv(J)`,
3. take part of that move (the fraction `alpha`), and repeat until the gap is below 0.1 mm.

The arm has six joints and the target only three numbers, so many joint moves would close the gap; the pseudo-inverse picks the smallest one. The solver is the whole loop. The pseudo-inverse is only step 2.

In [ ]:
def ik_ur5(target, q_init, alpha=0.5, tol=1e-4, max_iter=300):
    '''IK solver loop. Returns the final q, the q of every step, and the gap at every step.'''
    q = np.array(q_init, dtype=float)
    hist_q, hist_gap = [q.copy()], []
    for _ in range(max_iter):
        _, T = fk_ur5(q)
        gap = np.asarray(target) - T[:3, 3]            # 1. measure the gap
        hist_gap.append(np.linalg.norm(gap))
        if np.linalg.norm(gap) < tol:
            break
        dq = np.linalg.pinv(jacobian_pos(q)) @ gap      # 2. smallest joint move that closes it
        q = q + alpha * dq                              # 3. take part of it, repeat
        hist_q.append(q.copy())
    return q, np.array(hist_q), np.array(hist_gap)

target = np.array([0.35, 0.35, 0.45])
q_sol, hq, hgap = ik_ur5(target, q_home)
wrap = lambda a: (a + np.pi) % (2*np.pi) - np.pi
_, T = fk_ur5(q_sol)
print(f"converged in {len(hgap)} iterations, final gap {hgap[-1]:.1e} m")
print("q [deg]    :", np.rad2deg(wrap(q_sol)).round(1))
print("reached    :", T[:3, 3].round(4), "   target:", target)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogy(hgap, "-o", ms=3, color="#D85A30")
ax.set_xlabel("iteration"); ax.set_ylabel("gap to the target [m]")
ax.set_title("the gap shrinks with every pass of the loop"); ax.grid(alpha=.3)
plt.show()

plot_ur5(q_sol, target=target, title="UR5 at the IK answer (drag to rotate)").show()

### The solver in motion

Every frame below is one pass of the loop, from the home pose to the target (green diamond). Use the play button under the picture. A robot controller runs this same loop once per control cycle, 100 to 1000 times a second, and then the solver simply is the controller.

In [ ]:
frames = np.unique(np.linspace(0, len(hq) - 1, min(40, len(hq))).astype(int))
fig = plt.figure(figsize=(5.5, 4.8))
ax = fig.add_subplot(projection="3d")
arm_line, = ax.plot([], [], [], "-o", lw=4, ms=5, color="#333", markerfacecolor="#D85A30")
ax.scatter(*target, color="#2E9E5B", s=60, marker="D")
ax.set_xlim(-0.7, 0.7); ax.set_ylim(-0.7, 0.7); ax.set_zlim(0.0, 1.0)
ax.set_box_aspect((1.4, 1.4, 1.0))
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_zlabel("z [m]")
title = ax.set_title("")

def update(k):
    i = frames[k]
    p, _ = fk_ur5(hq[i])
    arm_line.set_data(p[:, 0], p[:, 1])
    arm_line.set_3d_properties(p[:, 2])
    title.set_text(f"solver step {i}, gap {np.linalg.norm(target - p[-1]) * 1000:.0f} mm")
    return arm_line, title

anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=250)
plt.close(fig)
HTML(anim.to_jshtml(default_mode="loop"))

### Closing the loop

Now the target runs around a circle and the arm follows it in small time steps. With `kp = 0` the joints move only as the circle says they should (this is called feedforward), so small errors pile up and nothing pulls the tool back. With `kp = 5` every step also pushes against the current gap, `kp` times the gap. Measuring the error and pushing against it is **feedback**, the same idea as the go-to-goal controller in Notebook 2.

In [ ]:
def track_circle(kp, T_total=4.0, dt=0.01, r=0.12, center=(0.4, 0.2, 0.5)):
    q = ik_ur5(np.array(center) + np.array([r, 0, 0]), q_home)[0]   # start on the circle
    errs = []
    for t in np.arange(0, T_total, dt):
        w = 2*np.pi / T_total
        p_star  = np.array([center[0] + r*np.cos(w*t), center[1], center[2] + r*np.sin(w*t)])   # where the tool should be
        pd_star = np.array([-r*w*np.sin(w*t), 0.0, r*w*np.cos(w*t)])                           # how fast it should move
        _, T = fk_ur5(q)
        gap = p_star - T[:3, 3]
        errs.append(np.linalg.norm(gap))
        q = q + dt * (np.linalg.pinv(jacobian_pos(q)) @ (pd_star + kp * gap))
    return np.arange(0, T_total, dt), np.array(errs)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for kp, col in [(0.0, "#888"), (5.0, "#D85A30")]:
    t, e = track_circle(kp)
    ax.plot(t, e * 1000, color=col, label=f"kp = {kp:g}")
ax.set_xlabel("time [s]"); ax.set_ylabel("tracking error [mm]")
ax.set_title("following a circle, without and with feedback")
ax.legend(); ax.grid(alpha=.3)
plt.show()

---
## What to take away

- A pose is a matrix, and chaining frames is matrix multiplication in the right order, `T = T @ T_i`. Matrix multiplication is not commutative, which is why the order inside a joint matters too: turn, then go along the link.
- A rotation matrix has determinant +1, and its transpose is its inverse. Determinant −1 would mean a mirror image.
- FK always has exactly one answer. IK can have two (elbow-down and elbow-up), one, or none (outside the workspace), and a closed-form formula exists only for special arms.
- In 3D, the DH parameters describe every joint with four numbers.
- The Jacobian has one column per joint and one row per direction of the tool. The IK solver is a loop, and the pseudo-inverse is the step inside it. With feedback added, the solver becomes a controller.
- All of this is explicit: every step uses a model written by hand. From week 4 the same problems are solved by learning from data.

Further reading: Lynch and Park, *Modern Robotics*, chapters 3 to 6; Hugging Face Robotics Course, Unit 2.